# TUGAS 4 UTS TEMU KEMBALI INFORMASI (SEMESTER GANJIL 2026/2027)
### Preprocessing, Extracting, Indexing (Inverted Index), dan Pembobotan Teks (TF-IDF & Okapi BM25) pada Seluruh Bagian Dokumen Putusan Narkotika PN Banjarmasin
**Program Studi Teknik Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang**

* **Mahasiswa**: Muhammad Aulia Rahman
* **NIM**: 202310370311005
* **Dosen Pengampu**: Galih W. W

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hiroyuuki01/Project-TKI/blob/main/Tugas4_Preprocessing_Indexing.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Project--TKI-blue.svg)](https://github.com/Hiroyuuki01/Project-TKI)

---

## 📌 Capaian Pembelajaran Penugasan (SUBCPMK-4)
> **SUBCPMK-4**: Mahasiswa mampu Memahami dan Menerapkan Model Temu Kembali Informasi dan Teknik Pembobotan dalam Pengindeksan dan Pencarian Teks.

Notebook ini mengimplementasikan tahapan terstruktur Information Retrieval (IR) pada korpus hukum **77 dokumen putusan perkara Pidana Khusus Narkotika Pengadilan Negeri Banjarmasin**:
1. **Pemuatan Seluruh Teks Dokumen Putusan** (Sesuai ketentuan Soal Tugas 4 poin b: *Gunakan seluruh bagian putusan sebagai dataset*).
2. **Pipeline Preprocessing Teks**: *Cleaning & Watermark Removal*, *Case Folding*, *Tokenizing*, *Normalisasi Istilah Hukum Narkotika*, *Stopword Removal* (Bahasa Indonesia & Istilah Struktural Hukum), dan *Fast Cached Stemming* (Sastrawi Nazief-Adriani).
3. **Pembangunan Inverted Index (Indeks Terbalik)** lengkap dengan *Posting List*, *Term Frequency* (TF), *Document Frequency* (DF), dan *Inverse Document Frequency* (IDF).
4. **Teknik Pembobotan Kata (Weighting)**: Mengimplementasikan **Sublinear TF-IDF (Vector Space Model)** dengan normalisasi L2 dan algoritma probabilistik **Okapi BM25** ($k_1=1.5, b=0.75$).
5. **Mesin Pencari Putusan (Query Retrieval)**: Menguji penemuan kembali dokumen putusan yang paling relevan berdasarkan kueri perkara narkotika nyata menggunakan *Cosine Similarity* dan *BM25 Ranking*.
6. **Visualisasi Distribusi Kata Kunci**: Menampilkan grafik frekuensi term paling dominan pada korpus putusan.

---
## Sel 1: Instalasi & Import Library
Memasang dan memuat pustaka yang dibutuhkan untuk pengolahan teks, komputasi matriks, dan visualisasi.

In [ ]:
import subprocess
import sys
import os

# Pastikan path site-packages .venv dikenali jika dijalankan di VS Code lokal
venv_site = os.path.abspath(os.path.join(os.getcwd(), ".venv", "Lib", "site-packages"))
if os.path.exists(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)

def install_packages():
    packages = ["PySastrawi", "pandas", "numpy", "matplotlib", "pdfplumber"]
    for pkg in packages:
        mod_name = "Sastrawi" if pkg == "PySastrawi" else pkg
        try:
            __import__(mod_name)
        except ImportError:
            try:
                print(f"Menginstal {pkg}...")
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "10", pkg]
                )
            except Exception:
                print(f"Info: {pkg} belum tersedia secara online, otomatis menggunakan modul internal.")

install_packages()

import re
import glob
import json
import math
import zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("[OK] Seluruh library berhasil dimuat dan siap digunakan!")

---
## Sel 2: Akses & Pemuatan Dataset (Seluruh Bagian Putusan)
Sesuai ketentuan **Soal Tugas 4 poin (a), (b), dan (e)**:
- Membaca teks putusan secara berurutan dari folder `TXT_Putusan` (atau `Folder_TXT_Putusan`).
- Sinkronisasi otomatis dari repositori GitHub jika dibuka dan dijalankan langsung di Google Colaboratory.
- Menyertakan *embedded fallback* sebagai pengaman agar tidak pernah mengalami `KeyError` atau crash pada lingkungan apapun.

In [ ]:
# Deteksi folder dataset lokal, subfolder Github, maupun Google Colab
POSSIBLE_TXT_DIRS = [
    "TXT_Putusan",
    "Folder_TXT_Putusan",
    os.path.join("Github", "TXT_Putusan"),
    os.path.join("Project-TKI", "TXT_Putusan"),
    os.path.join("..", "TXT_Putusan")
]
POSSIBLE_PDF_DIRS = [
    "PDF_Putusan",
    "Folder_PDF_Putusan",
    os.path.join("Github", "PDF_Putusan"),
    os.path.join("Project-TKI", "PDF_Putusan"),
    os.path.join("..", "PDF_Putusan")
]

TXT_DIR = next((d for d in POSSIBLE_TXT_DIRS if os.path.isdir(d) and len(glob.glob(os.path.join(d, "*.txt"))) > 0), "TXT_Putusan")
PDF_DIR = next((d for d in POSSIBLE_PDF_DIRS if os.path.isdir(d)), "PDF_Putusan")
CSV_FALLBACK = "Hasil_Ekstraksi_Tugas3.csv" if os.path.isfile("Hasil_Ekstraksi_Tugas3.csv") else os.path.join("Github", "Hasil_Ekstraksi_Tugas3.csv")

# Jika berjalan di Google Colab kosong dan folder TXT_Putusan belum ada, klon dataset dari GitHub
if not os.path.isdir(TXT_DIR) or len(glob.glob(os.path.join(TXT_DIR, "*.txt"))) == 0:
    print("[COLAB SYNC] Folder dataset lokal belum terdeteksi. Menyinkronkan korpus dari GitHub Repository...")
    try:
        subprocess.run(["git", "clone", "https://github.com/Hiroyuuki01/Project-TKI.git"], check=False)
        if os.path.isdir(os.path.join("Project-TKI", "TXT_Putusan")):
            TXT_DIR = os.path.join("Project-TKI", "TXT_Putusan")
            PDF_DIR = os.path.join("Project-TKI", "PDF_Putusan")
            CSV_FALLBACK = os.path.join("Project-TKI", "Hasil_Ekstraksi_Tugas3.csv")
            print(f"[BERHASIL] Sinkronisasi korpus putusan dari GitHub ke '{TXT_DIR}' selesai.")
    except Exception as err:
        print("[INFO] Clone git dilewati:", err)

# Ekstrak otomatis jika ada file zip di direktori kerja
for zf in glob.glob("*.zip"):
    if any(k in zf.lower() for k in ["txt", "putusan", "narkotika", "dataset"]):
        os.makedirs(TXT_DIR, exist_ok=True)
        with zipfile.ZipFile(zf, "r") as archive:
            archive.extractall(TXT_DIR)
        print(f"Berhasil mengekstrak '{zf}' ke '{TXT_DIR}'.")

# Sampel cadangan putusan asli PN Banjarmasin jika dijalankan di mesin kosong tanpa file
EMBEDDED_PN_BJM_SAMPLES = [
    {
        "no_putusan": "609/Pid.Sus/2026/PN Bjm",
        "full_text": (
            "Putusan Nomor 609/Pid.Sus/2026/PN Bjm Pengadilan Negeri Banjarmasin yang mengadili perkara pidana "
            "narkotika. Terdakwa I Wahyu Indra Laksana Bin Adriansyah Laksana dan Terdakwa II Herwanda Sumanan "
            "Bin Mursid. Menimbang bahwa Penuntut Umum mengajukan barang bukti berupa 1 paket narkotika jenis "
            "sabu dalam plastik klip berat bersih 5,12 gram, 1 bungkus rokok Sampoerna merah, 1 unit HP Infinix, "
            "dan 1 unit HP Oppo A3S. MENGADILI: Menyatakan para Terdakwa terbukti secara sah dan meyakinkan bersalah "
            "melakukan tindak pidana permufakatan jahat tanpa hak memiliki, menyimpan, menguasai narkotika golongan I "
            "bukan tanaman beratnya melebihi 5 gram melanggar Pasal 112 ayat (2) Jo Pasal 132 ayat (1) UU RI Nomor 35 "
            "Tahun 2009 tentang Narkotika. Menjatuhkan pidana penjara masing-masing 5 tahun dan denda Rp1.000.000.000 "
            "subsidair 3 bulan penjara. Menetapkan barang bukti sabu dirampas untuk dimusnahkan."
        )
    },
    {
        "no_putusan": "547/Pid.Sus/2026/PN Bjm",
        "full_text": (
            "Putusan Nomor 547/Pid.Sus/2026/PN Bjm Pengadilan Negeri Banjarmasin. Terdakwa Arbain Als Cabak Bin Isya. "
            "Tuntutan pidana Penuntut Umum meminta Majelis Hakim menyatakan terdakwa terbukti secara sah dan meyakinkan "
            "bersalah melakukan tindak pidana tanpa hak atau melawan hukum memiliki menyimpan menguasai Narkotika "
            "Golongan I bukan tanaman jenis sabu melanggar Pasal 112 ayat (1) UU RI Nomor 35 Tahun 2009 tentang Narkotika. "
            "Barang bukti berupa 1 paket serbuk kristal narkotika jenis sabu berat bersih 0,08 gram, 1 buah pipet kaca, "
            "dan korek api gas dirampas untuk dimusnahkan. MENGADILI: Menjatuhkan pidana penjara terhadap terdakwa "
            "selama 4 tahun dan denda Rp800.000.000."
        )
    },
    {
        "no_putusan": "573/Pid.Sus/2026/PN Bjm",
        "full_text": (
            "Putusan Nomor 573/Pid.Sus/2026/PN Bjm Pengadilan Negeri Banjarmasin. Terdakwa Eko Prasetyo Alias Eko Bin Sudaryanto. "
            "Menimbang bahwa terdakwa ditangkap di Kota Banjarmasin beserta barang bukti narkotika golongan I bukan "
            "tanaman jenis sabu. MENGADILI: Menyatakan terdakwa terbukti bersalah melakukan tindak pidana tanpa hak menjual "
            "membeli menjadi perantara dalam jual beli narkotika golongan I melanggar Pasal 114 ayat (1) UU Nomor 35 "
            "Tahun 2009. Menjatuhkan pidana penjara selama 6 tahun dan denda Rp1.000.000.000 subsidair 6 bulan penjara."
        )
    }
]

def load_full_decisions_dataset():
    documents = []
    
    # Peta metadata nomor putusan dari file CSV jika tersedia
    meta_no_putusan = {}
    if os.path.isfile(CSV_FALLBACK):
        try:
            df_meta = pd.read_csv(CSV_FALLBACK)
            for idx, r in df_meta.iterrows():
                fname = f"Putusan_{idx+1:02d}.txt"
                val = str(r.get("No Putusan", "")).strip()
                if val and val != "-":
                    meta_no_putusan[fname] = val
        except Exception:
            pass

    # 1. Baca seluruh teks putusan dari TXT_DIR
    txt_files = sorted(glob.glob(os.path.join(TXT_DIR, "*.txt")))
    if txt_files:
        print(f"Membaca {len(txt_files)} file teks putusan dari '{TXT_DIR}'...")
        for fpath in txt_files:
            fname = os.path.basename(fpath)
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                raw_text = f.read().strip()
            if len(raw_text) < 100:
                continue
            no_put = meta_no_putusan.get(fname, "")
            if not no_put:
                m_no = re.search(r"([0-9]{1,4}\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z\.]+)", raw_text, re.I)
                no_put = re.sub(r"\s+", " ", m_no.group(1)).strip().rstrip(".") if m_no else fname
            no_put = re.sub(r"PN\s*Bjm[a-zA-Z]*$", "PN Bjm", no_put, flags=re.I)
            documents.append({
                "doc_id": f"DOC_{len(documents)+1:02d}",
                "nama_file": fname,
                "no_putusan": no_put,
                "full_text": raw_text
            })

    # 2. Jika TXT kosong, coba baca dari PDF_DIR
    if not documents and os.path.isdir(PDF_DIR):
        pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
        if pdf_files:
            import pdfplumber
            print(f"Membaca {len(pdf_files)} file PDF dari '{PDF_DIR}'...")
            for fpath in pdf_files:
                fname = os.path.basename(fpath)
                pages = []
                with pdfplumber.open(fpath) as pdf:
                    for p in pdf.pages:
                        clean_p = p.filter(lambda obj: obj.get("object_type") != "char" or "Helvetica" not in obj.get("fontname", ""))
                        pages.append(clean_p.extract_text() or p.extract_text() or "")
                raw_text = "\n".join(pages).strip()
                if len(raw_text) < 100:
                    continue
                m_no = re.search(r"([0-9]{1,4}\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z\.]+)", raw_text, re.I)
                no_put = re.sub(r"\s+", " ", m_no.group(1)).strip().rstrip(".") if m_no else fname
                documents.append({
                    "doc_id": f"DOC_{len(documents)+1:02d}",
                    "nama_file": fname,
                    "no_putusan": no_put,
                    "full_text": raw_text
                })

    # 3. Fallback sampel jika di mesin cloud tanpa file
    if not documents:
        print("Menggunakan korpus sampel putusan PN Banjarmasin...")
        for idx, item in enumerate(EMBEDDED_PN_BJM_SAMPLES, start=1):
            documents.append({
                "doc_id": f"DOC_{idx:02d}",
                "nama_file": f"Putusan_{idx:02d}.txt",
                "no_putusan": item["no_putusan"],
                "full_text": item["full_text"]
            })

    df = pd.DataFrame(documents)
    df = df.drop_duplicates(subset=["no_putusan"], keep="first").reset_index(drop=True)
    df["doc_id"] = [f"DOC_{i+1:02d}" for i in range(len(df))]
    return df

df_corpus = load_full_decisions_dataset()
print(f"\n[BERHASIL] Total dokumen korpus putusan yang dimuat: {len(df_corpus)} dokumen.")
print(df_corpus[["doc_id", "nama_file", "no_putusan"]].head(10).to_string(index=False))

---
## Sel 3: Tahapan Preprocessing Teks Hukum Bahasa Indonesia
Menerapkan urutan tahapan pembersihan teks hukum secara komprehensif:
1. **Cleaning & Watermark Removal**: Menghapus nomor halaman berulang, URL, dan sisa karakter watermark vertikal MA.
2. **Case Folding**: Mengubah seluruh teks menjadi huruf kecil (*lowercase*).
3. **Tokenizing**: Memecah teks menjadi token-token kata individual.
4. **Normalisasi Istilah Hukum Narkotika**: Menyeragamkan varian kata (*shabu* $\rightarrow$ *sabu*, *als* $\rightarrow$ *alias*, *jpu* $\rightarrow$ *jaksa*, *extacy* $\rightarrow$ *ekstasi*, dll).
5. **Stopwords Removal**: Mengeliminasi kata tugas/hubung umum Bahasa Indonesia dan istilah struktural dokumen hukum.
6. **Fast Cached Stemming**: Mengubah kata berimbuhan ke kata dasar menggunakan **PySastrawi** (Nazief-Adriani) yang dioptimasi dengan *Dictionary Cache* (`stem_cache`) sehingga memproses korpus putusan hanya dalam hitungan detik.

In [ ]:
indonesian_stopwords = {
    "yang", "dan", "di", "ke", "dari", "pada", "dalam", "untuk", "dengan", "dan",
    "atau", "ini", "itu", "tersebut", "adalah", "ialah", "merupakan", "sebagai",
    "bahwa", "oleh", "karena", "jika", "apabila", "bila", "maka", "namun", "tetapi",
    "melainkan", "sedangkan", "serta", "ataupun", "maupun", "baik", "saja", "hanya",
    "juga", "pula", "lagi", "telah", "sudah", "sedang", "akan", "belum", "pernah",
    "tidak", "tak", "bukan", "tanpa", "jangan", "dapat", "bisa", "boleh", "harus",
    "wajib", "perlu", "ada", "adanya", "tidaklah", "kepada", "terhadap", "tentang",
    "mengenai", "secara", "seperti", "bagai", "bagaikan", "sebagaimana", "dimaksud",
    "antara", "hingga", "sampai", "sejak", "selama", "sebelum", "sesudah", "setelah",
    "ketika", "saat", "waktu", "para", "masing", "setiap", "tiap", "semua", "seluruh",
    "beberapa", "banyak", "sedikit", "lebih", "kurang", "paling", "sangat", "amat",
    "cukup", "agak", "terlalu", "begitu", "demikian", "demikianlah", "berikut",
    "diatas", "dibawah", "disini", "disana", "situ", "sini", "sana", "kami", "kita",
    "saya", "aku", "anda", "engkau", "kamu", "dia", "ia", "mereka", "beliau", "nya",
    "halaman", "putusan", "nomor", "mahkamah", "agung", "republik", "indonesia",
    "menimbang", "mengingat", "memperhatikan", "bjm", "wita", "wib", "wit",
    "tahun", "bulan", "hari", "tanggal", "pukul", "sekitar", "sekira", "jam",
    "jalan", "kelurahan", "kecamatan", "kota", "kabupaten", "provinsi", "bin", "binti"
}

try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    all_stopwords = indonesian_stopwords.union(set(StopWordRemoverFactory().get_stop_words()))
except ImportError:
    all_stopwords = indonesian_stopwords

legal_normalization_map = {
    "shabu": "sabu",
    "sabu-sabu": "sabu",
    "metamfetamina": "sabu",
    "extacy": "ekstasi",
    "exstacy": "ekstasi",
    "inex": "ekstasi",
    "als": "alias",
    "alm": "almarhum",
    "jpu": "jaksa",
    "uu": "undang",
    "uuri": "undang",
    "kuhap": "acara",
    "kuhp": "pidana",
    "bb": "bukti",
    "gr": "gram",
    "hp": "handphone",
    "subsidair": "subsider",
    "primair": "primer"
}

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _sastrawi_stemmer = StemmerFactory().create_stemmer()
except ImportError:
    _sastrawi_stemmer = None

def _fallback_indonesian_stem(word):
    """Algoritma Stemmer Morfologi Bahasa Indonesia (Nazief-Adriani Rule-based) berkecepatan tinggi."""
    if len(word) <= 4:
        return word
    # Partikel
    word = re.sub(r"(lah|kah|tah|pun)$", "", word)
    # Kata ganti kepunyaan
    if len(word) > 4:
        word = re.sub(r"(ku|mu|nya)$", "", word)
    # Awalan (Prefix)
    if len(word) > 5:
        word = re.sub(r"^(meng|meny|men|mem|me|peng|peny|pen|pem|pe|di|ter|ke|ber|per|se)", "", word)
    # Akhiran (Suffix)
    if len(word) > 4:
        word = re.sub(r"(kan|an|i)$", "", word)
    return word if len(word) >= 3 else word

stem_cache = {}
def cached_stem(word):
    if word not in stem_cache:
        if _sastrawi_stemmer is not None:
            stem_cache[word] = _sastrawi_stemmer.stem(word)
        else:
            stem_cache[word] = _fallback_indonesian_stem(word)
    return stem_cache[word]

def preprocess_legal_document(raw_text):
    # 1. Cleaning & Noise Removal
    text = re.sub(r"Halaman\s+\d+\s+dari\s+\d+[^\n]*", " ", str(raw_text), flags=re.I)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # 2. Case Folding
    text = text.lower()
    # Hapus angka dan simbol non-alfabet
    text = re.sub(r"[^a-z\s]", " ", text)
    # Hapus huruf tunggal sisa pecahan watermark
    text = re.sub(r"\b[a-z]\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 3. Tokenizing
    raw_tokens = text.split()

    # 4. Normalisasi Istilah & 5. Stopword Removal
    filtered_tokens = []
    for tok in raw_tokens:
        norm_tok = legal_normalization_map.get(tok, tok)
        if len(norm_tok) > 2 and norm_tok not in all_stopwords:
            filtered_tokens.append(norm_tok)

    # 6. Fast Cached Stemming
    stemmed_tokens = [cached_stem(tok) for tok in filtered_tokens]
    stemmed_tokens = [t for t in stemmed_tokens if len(t) > 2 and t not in all_stopwords]

    return {
        "clean_text": " ".join(stemmed_tokens),
        "tokens": stemmed_tokens,
        "raw_token_count": len(raw_tokens),
        "clean_token_count": len(stemmed_tokens)
    }

print("Memulai eksekusi Preprocessing pada seluruh dokumen putusan...")
preprocessed_results = [preprocess_legal_document(txt) for txt in df_corpus["full_text"]]

df_corpus["clean_text"] = [r["clean_text"] for r in preprocessed_results]
df_corpus["tokens"] = [r["tokens"] for r in preprocessed_results]
df_corpus["jumlah_kata_awal"] = [r["raw_token_count"] for r in preprocessed_results]
df_corpus["jumlah_kata_bersih"] = [r["clean_token_count"] for r in preprocessed_results]

print(f"\n[BERHASIL] Preprocessing selesai!")
print(f"Total kosakata unik yang di-stem pada cache: {len(stem_cache)} kata.")
print("\nPerbandingan Jumlah Kata Sebelum dan Sesudah Preprocessing (10 Dokumen Pertama):")
print(df_corpus[["doc_id", "no_putusan", "jumlah_kata_awal", "jumlah_kata_bersih"]].head(10).to_string(index=False))

---
## Sel 4: Pembangunan Inverted Index (Indeks Terbalik)
Membangun struktur data **Inverted Index** yang memetakan setiap kata dasar (*term*) ke daftar dokumen tempat kata tersebut muncul (*posting list*) beserta frekuensi kemunculannya (*Term Frequency* / TF), frekuensi dokumen (*Document Frequency* / DF), dan bobot *Inverse Document Frequency* (IDF).

In [ ]:
def build_inverted_index(df_data):
    inverted_index = defaultdict(dict)
    for _, row in df_data.iterrows():
        doc_id = row["doc_id"]
        term_counts = Counter(row["tokens"])
        for term, tf in term_counts.items():
            inverted_index[term][doc_id] = int(tf)

    index_records = []
    total_docs = len(df_data)
    for term, postings in inverted_index.items():
        df_val = len(postings)
        total_tf = sum(postings.values())
        idf_smooth = math.log((1 + total_docs) / (1 + df_val)) + 1.0
        posting_str = ", ".join([f"{d}:{c}" for d, c in sorted(postings.items())[:8]])
        if len(postings) > 8:
            posting_str += f", ... (+{len(postings)-8} dok)"
        index_records.append({
            "Term": term,
            "DF (Jumlah Dokumen)": df_val,
            "Total TF (Frekuensi Total)": total_tf,
            "IDF": round(idf_smooth, 4),
            "Posting List (DocID:TF)": posting_str
        })

    df_idx = pd.DataFrame(index_records).sort_values(
        by=["DF (Jumlah Dokumen)", "Total TF (Frekuensi Total)"],
        ascending=[False, False]
    ).reset_index(drop=True)
    return dict(inverted_index), df_idx

inverted_index_dict, df_inverted_index = build_inverted_index(df_corpus)

# Simpan Inverted Index ke file JSON dan CSV
with open("Inverted_Index_Tugas4.json", "w", encoding="utf-8") as f_json:
    json.dump(inverted_index_dict, f_json, ensure_ascii=False, indent=2)
df_inverted_index.to_csv("Tabel_Inverted_Index_Tugas4.csv", index=False, encoding="utf-8-sig")

print(f"[BERHASIL] Inverted Index terbentuk dengan total kosakata (Vocabulary): {len(df_inverted_index)} term unik.")
print(f"Tersimpan di: 'Inverted_Index_Tugas4.json' dan 'Tabel_Inverted_Index_Tugas4.csv'\n")
print("15 Term Teratas pada Inverted Index:")
print(df_inverted_index.head(15).to_string(index=False))

---
## Sel 5: Teknik Pembobotan Kata (TF-IDF Vector Space Model & Okapi BM25)
Menerapkan dua metode pembobotan teks utama dalam Information Retrieval (**SUBCPMK-4**):
1. **Sublinear TF-IDF (Vector Space Model)**: Menggunakan pembobotan sublinear $w_{t,d} = (1 + \ln \text{tf}_{t,d}) \times \left(\ln \frac{1 + N}{1 + \text{df}_t} + 1\right)$ disertai normalisasi L2 (*Cosine Similarity*).
2. **Okapi BM25**: Algoritma probabilistik yang memperhitungkan variasi panjang dokumen putusan dengan parameter standar $k_1 = 1.5$ dan $b = 0.75$.

In [ ]:
class SublinearTfidfEngine:
    """Mesin Vektor TF-IDF & Cosine Similarity murni berbasis NumPy."""
    def __init__(self, tokenized_docs):
        self.N = len(tokenized_docs)
        vocab_set = sorted({t for doc in tokenized_docs for t in doc})
        self.vocab = {t: i for i, t in enumerate(vocab_set)}
        self.feature_names = vocab_set
        df_counts = np.zeros(len(vocab_set), dtype=np.float64)
        tf_mat = np.zeros((self.N, len(vocab_set)), dtype=np.float64)

        for d_idx, tokens in enumerate(tokenized_docs):
            counts = Counter(tokens)
            for term, cnt in counts.items():
                t_idx = self.vocab[term]
                tf_mat[d_idx, t_idx] = 1.0 + math.log(cnt)
                df_counts[t_idx] += 1.0

        self.idf = np.log((1.0 + self.N) / (1.0 + df_counts)) + 1.0
        raw_tfidf = tf_mat * self.idf
        norms = np.linalg.norm(raw_tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self.tfidf_matrix = raw_tfidf / norms

    def score_query(self, query_tokens):
        q_vec = np.zeros(len(self.vocab), dtype=np.float64)
        counts = Counter(query_tokens)
        for term, cnt in counts.items():
            if term in self.vocab:
                t_idx = self.vocab[term]
                q_vec[t_idx] = (1.0 + math.log(cnt)) * self.idf[t_idx]
        q_norm = np.linalg.norm(q_vec)
        if q_norm > 0:
            q_vec = q_vec / q_norm
        return self.tfidf_matrix.dot(q_vec)

class OkapiBM25Index:
    """Implementasi Pembobotan Probabilistik Okapi BM25."""
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(tokenized_docs)
        self.doc_lens = [len(d) for d in tokenized_docs]
        self.avgdl = sum(self.doc_lens) / self.N if self.N > 0 else 1.0
        self.doc_freqs = []
        self.df = defaultdict(int)
        for tokens in tokenized_docs:
            counts = Counter(tokens)
            self.doc_freqs.append(counts)
            for term in counts.keys():
                self.df[term] += 1
        self.idf = {
            term: math.log(1.0 + (self.N - df_val + 0.5) / (df_val + 0.5))
            for term, df_val in self.df.items()
        }

    def score_query(self, query_tokens):
        scores = np.zeros(self.N)
        for term in query_tokens:
            if term not in self.idf:
                continue
            idf_val = self.idf[term]
            for idx, counts in enumerate(self.doc_freqs):
                tf = counts.get(term, 0)
                if tf > 0:
                    denom = tf + self.k1 * (1.0 - self.b + self.b * (self.doc_lens[idx] / self.avgdl))
                    scores[idx] += idf_val * (tf * (self.k1 + 1.0)) / denom
        return scores

print("Membangun matriks pembobotan TF-IDF dan indeks Okapi BM25...")
tfidf_engine = SublinearTfidfEngine(df_corpus["tokens"].tolist())
bm25_engine = OkapiBM25Index(df_corpus["tokens"].tolist(), k1=1.5, b=0.75)

print(f"[BERHASIL] Matriks TF-IDF selesai: {tfidf_engine.tfidf_matrix.shape[0]} Dokumen x {tfidf_engine.tfidf_matrix.shape[1]} Term Unik.")
print("[BERHASIL] Indeks Pembobotan Probabilistik Okapi BM25 siap digunakan!")

---
## Sel 6: Pengujian Mesin Pencari (Query Retrieval) & Evaluasi Ranking Dokumen
Mengevaluasi kemampuan temu kembali informasi dengan menguji kueri pencarian kasus narkotika nyata serta membandingkan skor relevansi antara **TF-IDF Cosine Similarity** dan **Okapi BM25**.

In [ ]:
def search_putusan(query, top_k=5):
    q_prep = preprocess_legal_document(query)
    q_clean = q_prep["clean_text"]
    q_tokens = q_prep["tokens"]

    tfidf_scores = tfidf_engine.score_query(q_tokens)
    bm25_scores = bm25_engine.score_query(q_tokens)

    df_res = pd.DataFrame({
        "Doc ID": df_corpus["doc_id"],
        "No Putusan": df_corpus["no_putusan"],
        "Nama File": df_corpus["nama_file"],
        "Skor TF-IDF Cosine": np.round(tfidf_scores, 4),
        "Skor Okapi BM25": np.round(bm25_scores, 4),
        "Cuplikan Teks Bersih": df_corpus["clean_text"].str.slice(0, 140) + "..."
    })

    df_top = df_res.sort_values(by=["Skor Okapi BM25", "Skor TF-IDF Cosine"], ascending=False).head(top_k).reset_index(drop=True)
    df_top.insert(0, "Peringkat", range(1, len(df_top) + 1))
    print(f"Query Asli        : '{query}'")
    print(f"Query Preprocessed: '{q_clean}'\n")
    return df_top

# Daftar Query Uji Relevansi Perkara Narkotika PN Banjarmasin
sample_queries = [
    "permufakatan jahat jual beli narkotika sabu berat melebihi 5 gram pasal 114 ayat 2",
    "barang bukti timbangan digital plastik klip sabu dan handphone dirampas untuk dimusnahkan",
    "menyimpan menguasai narkotika golongan I bukan tanaman pasal 112 ayat 1"
]

for i, q in enumerate(sample_queries, start=1):
    print("=" * 95)
    print(f"PENGUJIAN QUERY {i}:")
    display_df = search_putusan(q, top_k=5)
    print(display_df[["Peringkat", "Doc ID", "No Putusan", "Skor TF-IDF Cosine", "Skor Okapi BM25"]].to_string(index=False))
    print()

# Simpan hasil akhir Preprocessing & Ekstraksi ke CSV
df_export = df_corpus[["doc_id", "nama_file", "no_putusan", "jumlah_kata_awal", "jumlah_kata_bersih", "clean_text"]].copy()
df_export.to_csv("Hasil_Preprocessing_Tugas4.csv", index=False, encoding="utf-8-sig")
print("[OK] Berkas 'Hasil_Preprocessing_Tugas4.csv' berhasil disimpan!")

---
## Sel 7: Visualisasi Distribusi Kata Kunci (Top 15 Term Dominan)
Membuat visualisasi grafik batang term-term yang paling dominan muncul pada seluruh korpus putusan narkotika PN Banjarmasin.

In [ ]:
top_terms = df_inverted_index.head(15).iloc[::-1]
plt.figure(figsize=(10, 5))
plt.barh(top_terms["Term"], top_terms["Total TF (Frekuensi Total)"], color="#1e3a5f")
plt.xlabel("Total Frekuensi Kemunculan (TF)")
plt.ylabel("Kata Dasar (Stemmed Term)")
plt.title("15 Kata Kunci Paling Dominan pada Korpus Putusan Narkotika PN Banjarmasin")
plt.tight_layout()
plt.show()